# EDA Kualitas Udara

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
INPUT = ROOT / "hasil_cleaning_final" / "01_aqms_clean.csv"
if not INPUT.is_file():
    raise FileNotFoundError("Jalankan cleaning.ipynb terlebih dahulu.")
OUTPUT = ROOT / "hasil_analisis_moniq"
OUTPUT.mkdir(parents=True, exist_ok=True)
polutan = ["pm25_ug_m3", "pm10_ug_m3", "so2_ug_m3", "co_ug_m3", "o3_ug_m3", "no2_ug_m3"]
print("Sumber:", INPUT)


In [ ]:
df = pd.read_csv(INPUT, parse_dates=['tanggal'], low_memory=False)
print('Jumlah baris dan kolom:', df.shape)
print('Rentang tanggal:', df['tanggal'].min(), 'sampai', df['tanggal'].max())
print('Jumlah sensor:', df['sensor_id'].nunique())
display(df.head())


## Statistik polutan

In [ ]:
statistik = df[polutan].agg(['count','mean','median','std','min','max']).T.reset_index().rename(columns={'index':'polutan'})
statistik.to_csv(OUTPUT/'01_statistik_polutan.csv',index=False)
display(statistik.round(2))


In [ ]:
kelengkapan = pd.DataFrame({'polutan':polutan, 'nilai_terisi':[df[c].count() for c in polutan]})
kelengkapan['jumlah_baris'] = len(df)
kelengkapan['persen_terisi'] = (kelengkapan['nilai_terisi']/len(df)*100).round(2)
kelengkapan.to_csv(OUTPUT/'02_keterisian_polutan.csv',index=False)
display(kelengkapan)


## Rata-rata per stasiun

In [ ]:
kunci = ['tanggal','sensor_id','provinsi','kota_kabupaten']
harian_sensor = df.groupby(kunci,dropna=False)[polutan].mean().reset_index()
n_pm25 = df.groupby(kunci,dropna=False)['pm25_ug_m3'].count().reset_index(name='n_pm25_valid')
harian_sensor = harian_sensor.merge(n_pm25,on=kunci,how='left',validate='one_to_one')
harian_sensor.to_csv(OUTPUT/'03_harian_per_stasiun.csv',index=False)
display(harian_sensor.head())


## Tren harian

In [ ]:
tren_harian = harian_sensor.groupby('tanggal')[polutan].mean().reset_index()
tren_harian.to_csv(OUTPUT/'04_tren_harian.csv',index=False)
plt.figure(figsize=(11,4))
plt.plot(tren_harian['tanggal'],tren_harian['pm25_ug_m3'])
plt.title('Rata-rata harian PM2.5 pada stasiun teramati')
plt.ylabel('PM2.5 (µg/m³)');plt.xlabel('Tanggal');plt.tight_layout()
plt.savefig(OUTPUT/'grafik_tren_pm25.png',dpi=140);plt.show()


## Tren bulanan

In [ ]:
bulanan = harian_sensor.copy()
bulanan['bulan'] = bulanan['tanggal'].dt.to_period('M').astype(str)
tren_bulanan = bulanan.groupby('bulan')[polutan].mean().reset_index()
tren_bulanan.to_csv(OUTPUT/'05_tren_bulanan.csv',index=False)
display(tren_bulanan.tail())


## Perbandingan wilayah

In [ ]:
wilayah = harian_sensor.groupby(['provinsi','kota_kabupaten'],dropna=False)[polutan].mean().reset_index()
jumlah_stasiun = df.groupby(['provinsi','kota_kabupaten'],dropna=False)['sensor_id'].nunique().reset_index(name='jumlah_stasiun')
wilayah = wilayah.merge(jumlah_stasiun,on=['provinsi','kota_kabupaten'],validate='one_to_one')
wilayah.to_csv(OUTPUT/'06_ringkasan_polusi_wilayah_untuk_falisha.csv',index=False)
display(wilayah.sort_values('pm25_ug_m3',ascending=False).head(10))


## Grafik PM2.5

In [ ]:
top = wilayah.dropna(subset=['pm25_ug_m3']).nlargest(12,'pm25_ug_m3').sort_values('pm25_ug_m3')
plt.figure(figsize=(9,5))
plt.barh(top['kota_kabupaten'],top['pm25_ug_m3'])
plt.xlabel('Rata-rata PM2.5 (µg/m³)');plt.title('12 wilayah dengan rata-rata PM2.5 tertinggi (stasiun teramati)')
plt.tight_layout();plt.savefig(OUTPUT/'grafik_wilayah_pm25.png',dpi=140);plt.show()


## Konsentrasi tinggi

In [ ]:
p95 = df[polutan].quantile(0.95).reset_index()
p95.columns=['polutan','p95_ug_m3']
p95.to_csv(OUTPUT/'07_p95_polutan.csv',index=False)
display(p95.round(2))


In [ ]:
dashboard = harian_sensor.melt(id_vars=kunci+['n_pm25_valid'],value_vars=polutan,var_name='polutan',value_name='konsentrasi_ug_m3')
dashboard.to_csv(OUTPUT/'08_dashboard_polutan_harian.csv',index=False)
print('Jumlah baris dashboard:',len(dashboard))
display(dashboard.head())
